<a href="https://colab.research.google.com/github/nastya-kolle/transportation-problem-method-optimization/blob/main/MO_TransportationTask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from scipy.optimize import linprog

# Проверка через scipy
cost = np.array([
    [7, 8, 1, 2],
    [4, 5, 3, 8],
    [9, 2, 3, 6]
])

supply = np.array([200, 180, 190])
demand = np.array([150, 130, 150, 140])

m, n = cost.shape
c = cost.flatten()

# Ограничения: строки (supply)
A_eq = []
for i in range(m):
    row = np.zeros(m * n)
    row[i*n:(i+1)*n] = 1
    A_eq.append(row)
# Ограничения: столбцы (demand)
for j in range(n):
    col = np.zeros(m * n)
    col[j::n] = 1
    A_eq.append(col)

b_eq = np.concatenate([supply, demand])

res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=(0, None), method='highs')
x_opt = res.x.reshape(m, n)

print("Оптимальный план (scipy):")
print(x_opt)
print(f"Оптимальная стоимость: {res.fun:.1f}")

Оптимальный план (scipy):
[[  0.   0.  60. 140.]
 [150.   0.  30.   0.]
 [  0. 130.  60.   0.]]
Оптимальная стоимость: 1470.0


In [ ]:
import random


#  ВЫВОД

def print_matrix(M, name="M"):
    print(f"\n{name} ({len(M)}x{len(M[0])}):")
    for row in M:
        print("  " + " ".join(f"{v:>6}" for v in row))

def print_vector(v, name):
    print(f"{name}: {v}")


#  ВВОД / ГЕНЕРАЦИЯ

def input_int(prompt, lo=None):
    while True:
        try:
            x = int(input(prompt))
            if lo is not None and x < lo:
                print(f"Введите >= {lo}")
            else:
                return x
        except ValueError:
            print("Нужно целое число.")

def input_matrix(m, n):
    print(f"Введите матрицу тарифов: {m} строк по {n} чисел.")
    C = []
    for i in range(m):
        while True:
            parts = input(f"C[{i+1}] = ").split()
            if len(parts) == n:
                try:
                    C.append(list(map(int, parts)))
                    break
                except ValueError:
                    pass
            print(f"Нужно {n} целых чисел.")
    return C

def input_vector(k, name):
    print(f"Введите {name} ({k} чисел):")
    while True:
        parts = input(f"{name} = ").split()
        if len(parts) == k:
            try:
                return list(map(int, parts))
            except ValueError:
                pass
        print(f"Нужно {k} целых чисел.")

def random_cost(m, n):
    return [[random.randint(1, 20) for _ in range(n)] for _ in range(m)]

def random_balanced(m, n):
    a = [random.randint(5, 50) for _ in range(m)]
    b, rem = [], sum(a)
    for _ in range(n - 1):
        v = random.randint(0, rem)
        b.append(v)
        rem -= v
    b.append(rem)
    random.shuffle(b)
    return a, b


#  БАЛАНСИРОВКА

def balance(C, a, b):
    """Добавляет фиктивного поставщика или потребителя, если нужно."""
    Sa, Sb = sum(a), sum(b)
    if Sa == Sb:
        return C, a, b, "Задача сбалансирована."
    if Sa < Sb:
        diff = Sb - Sa
        return (C + [[0]*len(b)], a + [diff], b[:],
                f"Добавлен фиктивный поставщик (запас {diff}).")
    else:
        diff = Sa - Sb
        return ([row + [0] for row in C], a[:], b + [diff],
                f"Добавлен фиктивный потребитель (потребность {diff}).")


#  НАЧАЛЬНЫЙ ПЛАН

def northwest_corner(a, b):
    """
    Метод северо-западного угла.
    Начинаем с клетки (0,0) и двигаемся вправо-вниз.
    """
    A, B = a[:], b[:]
    X = [[0]*len(b) for _ in range(len(a))]
    i = j = 0
    while i < len(a) and j < len(b):
        x = min(A[i], B[j])
        X[i][j] = x
        A[i] -= x
        B[j] -= x
        if A[i] == 0 and B[j] == 0: j += 1          # вырожденный случай
        elif A[i] == 0: i += 1
        else: j += 1
    return X

def minimum_element(C, a, b):
    """
    Метод минимального элемента.
    Всегда берём клетку с наименьшим тарифом среди незакрытых.
    """
    A, B = a[:], b[:]
    X = [[0]*len(b) for _ in range(len(a))]

    # Сортируем все клетки по тарифу
    cells = sorted(
        [(C[i][j], i, j) for i in range(len(a)) for j in range(len(b))],
        key=lambda t: t[0]
    )

    for _, i, j in cells:
        if A[i] == 0 or B[j] == 0:
            continue                    # строка или столбец уже закрыты
        x = min(A[i], B[j])
        X[i][j] = x
        A[i] -= x
        B[j] -= x

    return X


#  БАЗИС И ВЫРОЖДЕННОСТЬ

def get_basics(X):
    """Возвращает список базисных клеток (где X[i][j] > 0)."""
    return [(i, j)
            for i in range(len(X))
            for j in range(len(X[0]))
            if X[i][j] > 0]

def would_create_cycle(basis, new_cell, m, n):
    """
    Проверка: создаст ли добавление new_cell цикл в дереве базиса
    """
    i0, j0 = new_cell
    visited_rows = {i0}
    visited_cols = set()

    changed = True
    while changed:
        changed = False
        for (i, j) in basis:
            if i in visited_rows and j not in visited_cols:
                visited_cols.add(j); changed = True
            elif j in visited_cols and i not in visited_rows:
                visited_rows.add(i); changed = True

    return j0 in visited_cols   # True => цикл


def ensure_nondegenerate(X, basics, m, n):
    """
    В невырожденном базисе должно быть ровно m+n-1 клеток.
    Если меньше — добавляем клетки с нулём (не создающие цикл).
    """
    basis = list(basics)
    need  = m + n - 1

    for i in range(m):
        for j in range(n):
            if len(basis) >= need:
                return basis
            if (i, j) not in basis and not would_create_cycle(basis, (i, j), m, n):
                basis.append((i, j))

    return basis


#  МЕТОД ПОТЕНЦИАЛОВ (MODI)

def compute_potentials(C, basics, m, n):
    u = [None] * m
    v = [None] * n
    u[0] = 0

    changed = True
    while changed:
        changed = False
        for (i, j) in basics:
            if u[i] is not None and v[j] is None:
                v[j] = C[i][j] - u[i];  changed = True
            elif v[j] is not None and u[i] is None:
                u[i] = C[i][j] - v[j];  changed = True

    # Страховка на случай несвязных компонент
    u = [x if x is not None else 0 for x in u]
    v = [x if x is not None else 0 for x in v]
    return u, v


def reduced_costs(C, u, v):
    """Оценки свободных клеток: delta[i][j] = C[i][j] - u[i] - v[j]."""
    return [[C[i][j] - u[i] - v[j] for j in range(len(C[0]))]
            for i in range(len(C))]


def find_entering(delta, basis_set):
    """Выбираем свободную клетку с наименьшей (наиболее отрицательной) оценкой."""
    best, best_val = None, 0
    for i, row in enumerate(delta):
        for j, d in enumerate(row):
            if (i, j) not in basis_set and d < best_val:
                best, best_val = (i, j), d
    return best


def find_cycle(basis_set, enter, m, n):
    """
    Строим цикл пересчёта
    """
    all_cells = basis_set | {enter}

    # Для быстрого поиска: какие столбцы доступны в строке i, и наоборот
    row_cols = {}   # row_cols[i] = список столбцов j таких что (i,j) в базисе
    col_rows = {}   # col_rows[j] = список строк i таких что (i,j) в базисе
    for (i, j) in all_cells:
        row_cols.setdefault(i, []).append(j)
        col_rows.setdefault(j, []).append(i)

    def dfs(path, by_row):
        """path — текущая цепь клеток; by_row — двигаемся по строке или столбцу."""
        i, j  = path[-1]
        # Список клеток-кандидатов для следующего шага
        if by_row:
            candidates = [(i, jj) for jj in row_cols.get(i, []) if jj != j]
        else:
            candidates = [(ii, j) for ii in col_rows.get(j, []) if ii != i]

        for nxt in candidates:
            if nxt == enter and len(path) >= 4:
                return path              # цикл замкнулся
            if nxt not in path:
                result = dfs(path + [nxt], not by_row)
                if result:
                    return result
        return None

    # Пробуем оба направления старта
    return dfs([enter], True) or dfs([enter], False)


#  СТОИМОСТЬ И ОСНОВНОЙ ЦИКЛ ОПТИМИЗАЦИИ

def total_cost(C, X):
    return sum(C[i][j] * X[i][j]
               for i in range(len(C)) for j in range(len(C[0])))


def transport_modi(C, X0, max_iter=1000, verbose=True):
    m, n = len(C), len(C[0])
    X = [row[:] for row in X0]

    basics    = get_basics(X)
    print(basics, "Old basis cells")
    basics    = ensure_nondegenerate(X, basics, m, n)
    print(basics, "New basis cells")
    basis_set = set(basics)

    for iteration in range(1, max_iter + 1):

        # Считаем потенциалы и оценки
        u, v  = compute_potentials(C, list(basis_set), m, n)
        delta = reduced_costs(C, u, v)

        # Ищем входящую переменную
        enter = find_entering(delta, basis_set)
        if enter is None:
            break                        # все оценки >= 0 => оптимум

        if verbose:
            i0, j0 = enter
            print(f"\nИтерация {iteration}: вводим {enter},  Δ = {delta[i0][j0]}")

        # Строим цикл пересчёта
        cycle = find_cycle(basis_set, enter, m, n)
        if verbose:
            print("Цикл: " + " → ".join(str(c) for c in cycle))

        # θ = минимум в «минусовых» клетках (нечётные позиции цикла)
        minus_cells = cycle[1::2]
        theta = min(X[i][j] for (i, j) in minus_cells)
        if verbose:
            print(f"θ = {theta}")

        # Пересчёт плана
        for k, (i, j) in enumerate(cycle):
            X[i][j] += theta if k % 2 == 0 else -theta

        # Обновляем базис: вводим enter, выводим клетку ставшую нулём
        basis_set.add(enter)
        leaving = next((c for c in minus_cells if X[c[0]][c[1]] == 0), None)
        if leaving is None:
            leaving = min(minus_cells, key=lambda c: X[c[0]][c[1]])
        basis_set.remove(leaving)

        if verbose:
            print(f"Выводим {leaving},  Z = {total_cost(C, X)}")

    return X, total_cost(C, X)


#  ТОЧКА ВХОДА
print("=== Транспортная задача ===\n")

m = input_int("Число поставщиков m: ", lo=1)
n = input_int("Число потребителей n: ", lo=1)

mode = input("Режим (manual/random) [random]: ").strip().lower() or "random"

if mode == "manual":
    C = input_matrix(m, n)
    a = input_vector(m, "запасы a")
    b = input_vector(n, "потребности b")
else:
    C = random_cost(m, n)
    a, b = random_balanced(m, n)

print_matrix(C, "Тарифы C")
print_vector(a, "Запасы a")
print_vector(b, "Потребности b")
print(f"  sum(a) = {sum(a)},  sum(b) = {sum(b)}")

# Балансировка
C, a, b, msg = balance(C, a, b)
print(f"\n{msg}")

# Начальный план (выбери один из двух методов)
method = input("\nНачальный план: (nw) северо-западный угол / (me) минимальный элемент [nw]: ").strip().lower() or "nw"
if method == "me":
    X = minimum_element(C, a, b)
    print_matrix(X, "Начальный план (мин. элемент)")
else:
    X = northwest_corner(a, b)
    print_matrix(X, "Начальный план (северо-западный угол)")

print(f"Z_нач = {total_cost(C, X)}")

# Оптимизация
print("\n--- Оптимизация методом потенциалов ---")
X_opt, Z_opt = transport_modi(C, X, verbose=True)

print_matrix(X_opt, "Оптимальный план")
print(f"\nZ_min = {Z_opt}")


=== Транспортная задача ===

Число поставщиков m: 3
Число потребителей n: 4
Режим (manual/random) [random]: manual
Введите матрицу тарифов: 3 строк по 4 чисел.
C[1] = 1 7 4 8
C[2] = 2 9 3 5
C[3] = 6 2 9 1
Введите запасы a (3 чисел):
запасы a = 150 180 190
Введите потребности b (4 чисел):
потребности b = 150 80 150 140

Тарифы C (3x4):
       1      7      4      8
       2      9      3      5
       6      2      9      1
Запасы a: [150, 180, 190]
Потребности b: [150, 80, 150, 140]
  sum(a) = 520,  sum(b) = 520

Задача сбалансирована.

Начальный план: (nw) северо-западный угол / (me) минимальный элемент [nw]: nw

Начальный план (северо-западный угол) (3x4):
     150      0      0      0
       0     80    100      0
       0      0     50    140
Z_нач = 1760

--- Оптимизация методом потенциалов ---
[(0, 0), (1, 1), (1, 2), (2, 2), (2, 3)] Old basis cells
[(0, 0), (1, 1), (1, 2), (2, 2), (2, 3), (0, 1)] New basis cells

Итерация 1: вводим (2, 1),  Δ = -13
Цикл: (2, 1) → (2, 2) → (1, 2)